# AIC2026 — sinh caption tieng Viet cho keyframe (KENH 5)

Doc kem: `docs/17_kaggle_encode_va_caption.md` muc VIEC B.

## Settings BAT BUOC

| | |
| --- | --- |
| Accelerator | **GPU T4 x2** |
| Internet | **ON** |
| Input | `aic2026-index` + 9 dataset anh |

⚠️ **DUNG chon P100** (sm_60) — torch tren Kaggle chi build cho sm_70 tro len.

## Hai dieu quyet dinh ket qua, doc truoc khi chay

**1. Caption PHAI la tieng Viet.** BM25 khop MAT CHU. Caption tieng Anh +
truy van tieng Viet = khop dung 0 token, kenh im lang tra ve rong. Kenh 1 da
duoc **0,0000** tren tap dev vi CLIP mu tieng Viet (A10) — dung dung lai cung
mot loi.

**2. Quet TRON VEN cac video duoc giao, DUNG chon quanh dap an.** BM25 chi xep
hang tai lieu CO TON TAI. Caption rieng khung dap an la dung mot be ung vien ma
dap an chiem phan lon — kenh se tim ra dap an vi no gan nhu la thu duy nhat co
tai lieu, khong phai vi caption ta dung. Do duoc tren file dau tien: **55% anh
da caption la khung dap an**, phep do do vut di.

## 1. Ma nguon

In [ ]:
!rm -rf /tmp/repo
!git clone -q -b giai-doan-0 https://github.com/QuocKhanhDev-it/AIC_2026_FirstDance.git /tmp/repo
%cd /tmp/repo
!pip -q install -U transformers accelerate qwen-vl-utils pandas pyarrow

## 2. Chot GPU — chay TRUOC khi tai trong so

In [ ]:
import torch
assert torch.cuda.is_available(), "khong co GPU — kiem Settings > Accelerator"
ten = torch.cuda.get_device_name(0)
cc = torch.cuda.get_device_capability(0)
ho_tro = torch.cuda.get_arch_list()
print(f"GPU: {ten}  sm_{cc[0]}{cc[1]}")
assert f"sm_{cc[0]}{cc[1]}" in ho_tro, (
    f"{ten} (sm_{cc[0]}{cc[1]}) khong nam trong build torch {ho_tro}. "
    "Doi Accelerator sang T4 x2.")
print("OK")

## 3. Chep `master.parquet` va va duong dan

`kf_path` la duong dan tuyet doi cua may dung index (`D:\Project\...`).
Bo qua buoc va thi script thay **khong co anh nao** va khong sinh gi ca.

In [ ]:
import glob, shutil, os, pathlib
print("co trong /kaggle/input:", os.listdir('/kaggle/input'))
pathlib.Path('index').mkdir(exist_ok=True)
hit = glob.glob('/kaggle/input/**/master.parquet', recursive=True)
assert hit, "khong thay master.parquet — da Add Input dataset aic2026-index chua?"
shutil.copy(hit[0], 'index/master.parquet')
print("chep tu", hit[0])

In [ ]:
!python scripts/12_va_duong_dan.py --roots /kaggle/input --ghi

### Ham `chay()` — `!lenh` that bai KHONG lam dung notebook

In [ ]:
import subprocess

def chay(lenh):
    print("$", lenh, flush=True)
    p = subprocess.run(lenh, shell=True, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(p.stdout)
    if p.returncode != 0:
        raise RuntimeError(f"ma thoat {p.returncode}: {lenh}")
    return p.stdout

### Chot chan — GOI TEN nhom con thieu

In [ ]:
import pandas as pd
MONG_DOI = {'L21': 7800, 'L22': 9096, 'L23': 2326, 'L24': 6781, 'L25': 37445,
            'L27': 4914, 'L28': 10683, 'L29': 10771, 'L30': 7915}
m = pd.read_parquet('index/master.parquet')
co = m[m.kf_path.notna()].video_id.str[:3].value_counts().to_dict()
thieu = [n for n in MONG_DOI if co.get(n, 0) == 0]
for n, c in sorted(MONG_DOI.items()):
    print(f"  {n}  {int(co.get(n, 0)):>7,} / {c:>7,}")
assert not thieu, f"CHUA Add Input cho nhom: {', '.join(thieu)}"
print("\nDu ca 9 nhom.")

## 4. Phan cua toi — DOI DUNG MOT SO

Dung **cung phep chia** voi viec encode (`scripts/47_chia_viec_encode.py`), nen
neu ca nhom chia 6 phan thi phan so N o day chinh la phan so N ben encode.

97.731 anh / 6 = ~16.290 anh moi phan. Nhung caption cham hon encode nhieu —
xem cell do toc do o muc 5 truoc khi cam ket.

Lam mot minh thi de `SO_NGUOI = 1` va `PHAN_CUA_TOI = 1` (ca 97.731 anh).

In [ ]:
PHAN_CUA_TOI = 1        # <-- DOI SO NAY
SO_NGUOI = 6            # de 1 neu muon lam het trong mot phien

import importlib.util, pandas as pd
_s = importlib.util.spec_from_file_location(
    "chia_viec", "scripts/47_chia_viec_encode.py")
_cv = importlib.util.module_from_spec(_s); _s.loader.exec_module(_cv)

m = pd.read_parquet('index/master.parquet')
phan, tai = _cv.chia(m, SO_NGUOI)
v = phan[PHAN_CUA_TOI - 1]
TEN_DS = f'phan_{PHAN_CUA_TOI}.txt'
open(TEN_DS, 'w').write('\n'.join(v) + '\n')
print(f"phan {PHAN_CUA_TOI}/{SO_NGUOI}: {len(v)} video, {tai[PHAN_CUA_TOI-1]:,} anh")

## 5. DO TOC DO THAT truoc — 40 anh

`1,5 giay/anh` trong bang uoc tinh la con so **CHUA DO** tren phan cung Kaggle.
Chay 40 anh, doc dong `giay/anh` that, roi nhan len moi quyet dinh pham vi.

Lan tai model dau mat them vai phut (~6 GB trong so Qwen2.5-VL-3B).

In [ ]:
chay(f"python scripts/14_sinh_caption.py --backend hf --chon video:{TEN_DS} "
     f"--n 40 --batch 8")

Nhan `giay/anh` doc duoc voi so anh cua phan minh:

* duoi 1,0 giay/anh -> ca phan ~4,5 gio, chay duoc trong mot phien
* tren 2,5 giay/anh -> vuot 11 gio, phai chia nho hon hoac ha `--diem-anh`

Hai nut tang toc, ca hai deu KHONG phai `--batch`:

| nut | y nghia |
| --- | --- |
| `--diem-anh` | tran token thi giac moi anh. **Dat nhat.** 512 -> 256 nhanh gan gap doi, doi lai model nhin anh mo hon |
| `--so-chu` | tran token sinh ra. Caption 2-3 cau khong can qua 180 |

Tran VRAM thi ha `--batch` xuong 4, roi moi ha `--diem-anh`.

## 6. Chay that

`caption.jsonl` ghi NOI tung dong — phien bi cat van giu nguyen phan da lam,
chay lai la bo qua anh da xong. Nen cell nay chay lai duoc bao nhieu lan cung
duoc.

In [ ]:
chay(f"python scripts/14_sinh_caption.py --backend hf --chon video:{TEN_DS} "
     f"--batch 8 --diem-anh 512 --so-chu 180")

In [ ]:
import shutil
for f in ('caption.jsonl', 'caption.parquet'):
    shutil.copy(f'index/{f}', '/kaggle/working/')
print("da chep sang /kaggle/working")

## 7. Doc caption bang MAT truoc khi bao xong

Ba thu lam caption thanh vo dung ma MOI kiem tra cau truc deu xanh:

* **Lan chu Han/Nhat/Han**. Do duoc o `qwen2.5vl:7b`: ~12% caption, va KHONG
  ngau nhien — don vao mot loai canh (ban dan chuong trinh tin tuc) lap qua
  nhieu khung lien tiep.
* **Lap vo han**. Tung gap mot caption 9.872 ky tu lap cung mot cau hang tram
  lan. `--so-chu 180` chan tran, nhung caption dai bat thuong van dang nghi.
* **Caption tieng Anh**. Khop dung 0 token voi truy van tieng Viet.

In [ ]:
import pandas as pd, re
d = pd.read_parquet('index/caption.parquet')
han = re.compile(r"[\u4e00-\u9fff\u3040-\u30ff\uac00-\ud7af]")
print(f"{len(d):,} caption")
print(d.caption.str.len().describe()[['min','25%','50%','75%','max']].astype(int).to_string())
lan = d.caption.str.contains(han, regex=True)
print(f"\nlan chu Han/Nhat/Han: {int(lan.sum())}/{len(d)} ({lan.mean()*100:.1f}%)")
dai = d[d.caption.str.len() > 700]
print(f"caption dai bat thuong (>700 ky tu): {len(dai)}")
print("\n10 caption ngau nhien:")
for c in d.caption.sample(min(10, len(d)), random_state=0):
    print(" -", c[:160])